---------

Comments
- 9% of location data are missing, but:
    - For active users we do not need location data
    - The missing data will average out (likely the 9% is MCAR)

--------
# Barter Deals dataset


- Construct the main predictor $apps\_after\_7\_days$
- Construct MAU/WAU (monthly and weekly active users, based on how many creators applied to a deal)
- Use location data to get estimates on the nr. of 'eligible' creators: the number of creators that are withina certain range of a (physical) deal

In [1]:
import numpy as np
import pandas as pd
from src import paths

In [2]:
df_apps = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEAL_APPLICATIONS.parquet')
df_deals = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEALS_CLEAN.parquet')

In [3]:
df_apps

,legacy_location_id,deal_id,creator_id,rescheduling_deal_application_id,schedule_date,application_status,cancelled_at,cancelled_by,cancelled_by_user_id,cancellation_reason,...,email_address_partners,registered_at_partners,subscription_status_partners,subscription_start_date_partners,subscription_end_date_partners,deals_count_partners,live_deals_count_partners,locations_count_partners,company_size_partners,worked_with_creators_partners
0,335,019689e3-9b82-00c8-52df-2e22979a06ee,8162,<NA>,2024-04-12 00:00:00+00:00,rejected,NaT,<NA>,NaN,<NA>,...,haleigh.feest@anonymized.getbarter.com,2024-02-15 14:31:26.387130+00:00,active,2024-02-15 14:31:21+00:00,2026-03-16 14:31:21+00:00,2.0,0.0,2.0,<NA>,<NA>
1,335,019689e3-9b82-00c8-52df-2e22979a06ee,7787,<NA>,2024-03-26 00:00:00+00:00,cancelled,NaT,<NA>,NaN,<NA>,...,haleigh.feest@anonymized.getbarter.com,2024-02-15 14:31:26.387130+00:00,active,2024-02-15 14:31:21+00:00,2026-03-16 14:31:21+00:00,2.0,0.0,2.0,<NA>,<NA>
2,335,019689e3-9b82-00c8-52df-2e22979a06ee,7827,<NA>,2024-05-29 00:00:00+00:00,accepted,NaT,<NA>,NaN,<NA>,...,haleigh.feest@anonymized.getbarter.com,2024-02-15 14:31:26.387130+00:00,active,2024-02-15 14:31:21+00:00,2026-03-16 14:31:21+00:00,2.0,0.0,2.0,<NA>,<NA>
3,335,019689e3-9b82-00c8-52df-2e22979a06ee,358,<NA>,2024-07-19 00:00:00+00:00,accepted,NaT,<NA>,NaN,<NA>,...,haleigh.feest@anonymized.getbarter.com,2024-02-15 14:31:26.387130+00:00,active,2024-02-15 14:31:21+00:00,2026-03-16 14:31:21+00:00,2.0,0.0,2.0,<NA>,<NA>
4,335,019689e3-9b82-00c8-52df-2e22979a06ee,343,<NA>,2024-05-20 00:00:00+00:00,accepted,NaT,<NA>,NaN,<NA>,...,haleigh.feest@anonymized.getbarter.com,2024-02-15 14:31:26.387130+00:00,active,2024-02-15 14:31:21+00:00,2026-03-16 14:31:21+00:00,2.0,0.0,2.0,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269729,102,019689e4-4bc6-00c8-6ff0-06aca342b058,1577,<NA>,2024-06-29 00:00:00+00:00,finished,NaT,<NA>,NaN,<NA>,...,marlene@anonymized.getbarter.com,2023-10-06 14:09:42.860289+00:00,active,2026-02-17 11:16:02+00:00,2027-02-17 11:31:14+00:00,4.0,1.0,1.0,<NA>,<NA>
269730,102,019689e4-4bc6-00c8-6ff0-06aca342b058,1577,<NA>,2024-07-06 00:00:00+00:00,finished,NaT,<NA>,NaN,<NA>,...,marlene@anonymized.getbarter.com,2023-10-06 14:09:42.860289+00:00,active,2026-02-17 11:16:02+00:00,2027-02-17 11:31:14+00:00,4.0,1.0,1.0,<NA>,<NA>
269731,<NA>,019b359e-7880-00c8-ba13-e73e44044ea9,36953,<NA>,2025-12-22 23:00:00+00:00,accepted,NaT,<NA>,NaN,<NA>,...,watsica.jaiden@anonymized.getbarter.com,2025-12-04 15:26:35.524300+00:00,active,2025-12-19 08:01:56+00:00,2026-03-24 18:39:37+00:00,2.0,1.0,1.0,xs,True
269732,<NA>,019b6abb-9953-00c8-168a-f649e05f64ba,34252,<NA>,NaT,accepted,NaT,<NA>,NaN,<NA>,...,watsica.jaiden@anonymized.getbarter.com,2025-12-04 15:26:35.524300+00:00,active,2025-12-19 08:01:56+00:00,2026-03-24 18:39:37+00:00,2.0,1.0,1.0,xs,True


# Generate metrics

## Nr of active creators 


In [ ]:
df_apps.craetor_id

In [5]:
import pandas as pd
from collections import defaultdict

# 1. Sort the DataFrame by date (CRITICAL for a sliding window)
df_apps = df_apps.sort_values('application_created_at').reset_index(drop=True)

# 2. Extract to NumPy arrays for lightning-fast iteration
# Converting datetimes to nanoseconds makes comparisons incredibly fast
dates = df_apps['application_created_at'].astype('int64').values
influencers = df_apps['creator_id'].values

# 3. Define your time windows in nanoseconds
ns_30_days = pd.Timedelta(days=30).value
ns_7_days = pd.Timedelta(days=7).value

def get_rolling_uniques(dates, influencers, window_ns):
    n = len(dates)
    results = [0] * n
    
    counts = defaultdict(int)
    unique_count = 0
    
    left = 0
    add_ptr = 0
    
    for i in range(n):
        current_time = dates[i]
        min_time = current_time - window_ns
        
        # EXPAND WINDOW: Add elements strictly LESS than the current row's time
        while add_ptr < n and dates[add_ptr] < current_time:
            inf = influencers[add_ptr]
            if counts[inf] == 0:
                unique_count += 1
            counts[inf] += 1
            add_ptr += 1
            
        # SHRINK WINDOW: Remove elements that fall outside the trailing window
        while left < add_ptr and dates[left] <= min_time:
            inf = influencers[left]
            counts[inf] -= 1
            if counts[inf] == 0:
                unique_count -= 1
            left += 1
            
        # Record the active unique count for this specific row
        results[i] = unique_count
        
    return results

# 4. Apply the optimized function
df_apps['active_last_month'] = get_rolling_uniques(dates, influencers, ns_30_days)
df_apps['active_last_week'] = get_rolling_uniques(dates, influencers, ns_7_days)

## Apps after 7 days

In [6]:
def apps_after_n_days(row, n=7):
    subs = df_apps[df_apps['deal_id'] == row['deal_id']]
    if len(subs) != row['applicants_applications_count']:
        subs = subs[subs['deleted_at'].isna()]

    # "created_at" is the date on which the application log entry was created (application was made)
    days_since_live = (subs.application_created_at - row.live_since).dt.days
    return ((days_since_live >= 0) & (days_since_live <= n)).sum()

apps_after_7_days = df_deals.apply(apps_after_n_days, axis=1)
df_deals['apps_after_7_days'] = apps_after_7_days


In [7]:
df_deals

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,created_at,updated_at,deleted_at,...,legacy_id,tags,gender,featured_image,company_id,test_nr_of_apps,update_at_date,diff,apps_after_7_days,apps_after_7_days_OLD
0,39,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",0196edc7-e6a8-00c8-6ff5-ea60fa3f2895,uploads/deals/0196edc7-5969-ffff-6700-7ae699b8...,2500,None,2025-05-20 15:21:28.249567,2025-05-20 13:00:23.080212,2025-10-28 10:55:34.719187,NaT,...,NaN,None,None,None,0199f1d4-b501-015e-5c45-1bc8300b3949,39,2025-10-28,0,39,39.0
1,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e5-f489-00c8-8c26-06b3ea0961a5,uploads/deals/019689e5-f54b-ffff-dbed-637e9d2a...,5000,None,2023-11-28 14:59:51.626780,2023-11-28 14:59:51.626780,2025-05-01 03:31:11.600095,NaT,...,487.0,None,None,None,None,0,2025-05-01,0,0,0.0
2,23,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019689e2-bfb5-00c8-6485-23352d64b0e5,uploads/deals/019689e2-c06c-ffff-baf0-c7e9decd...,5000,None,2025-03-04 11:46:18.176627,2025-03-04 11:46:18.176627,2025-05-01 03:27:41.316646,NaT,...,3593.0,None,None,None,None,23,2025-05-01,0,13,13.0
3,43,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e3-d7a5-00c8-f5a8-6bfe51a21cd6,uploads/deals/019689e3-d86f-ffff-a03e-d38765a0...,2500,None,2025-04-17 09:50:45.518632,2025-04-17 09:50:45.518632,2025-06-08 09:05:21.413273,NaT,...,4228.0,None,None,None,None,43,2025-06-08,0,19,19.0
4,13,"[{'id': 19, 'name': 'Sport', 'slug': 'Soccer-B...",019689e5-7dd0-00c8-615b-7750a8fec2f3,uploads/deals/019689e5-7f9d-ffff-5b6d-a335c1ce...,2500,None,2025-02-10 18:19:51.240203,2025-02-10 18:19:51.240203,2025-10-28 10:51:05.147656,NaT,...,3242.0,None,None,None,0199f1b0-2b6c-015e-e5b3-6b1dd7564d3a,13,2025-10-28,0,13,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5106,9,"[{'id': 39, 'name': 'Experiences', 'slug': 'Te...",019689e5-4642-00c8-08dd-41d776586f1c,uploads/deals/019689e5-46b6-ffff-2159-fb70a484...,50000,None,2024-03-26 14:09:02.539962,2024-03-26 14:09:02.539962,2025-10-28 10:54:41.617599,NaT,...,865.0,None,None,None,0199f1cc-e3f7-015e-432a-d2035c7ba015,9,2025-10-28,0,5,5.0
5107,11,"[{'id': 39, 'name': 'Experiences', 'slug': 'Te...",019689e5-6ce0-00c8-5fdd-a058811d2946,uploads/deals/019689e5-6e0d-ffff-4a2f-362bb635...,10000,None,2024-10-17 14:59:27.484092,2024-10-17 14:59:27.484092,2025-10-28 10:52:42.692581,NaT,...,1889.0,None,None,None,0199f1c3-fcaf-015e-eeb9-cc244704412b,11,2025-10-28,0,7,7.0
5108,0,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}]",0198324e-635d-00c8-7d1d-bbb0b902b0d9,,10000,None,NaT,2025-07-22 13:24:14.813323,2025-10-28 11:04:12.121141,NaT,...,NaN,None,None,None,0199f18e-af17-015e-23aa-c780e5c4a2b8,0,2025-10-28,0,0,NaN
5109,0,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",01983c98-87ee-00c8-e551-8fd9afde13aa,,5000,None,NaT,2025-07-24 13:21:25.998303,2025-10-28 10:51:32.951354,NaT,...,NaN,None,None,None,0199f1b6-5645-015e-ffbf-5e1786a1e5c3,0,2025-10-28,0,0,NaN


creator_id
10525    3067
9905     2043
4973     1913
5780     1627
7611     1597
         ... 
29746       1
21627       1
3241        1
6245        1
3930        1
Name: count, Length: 6696, dtype: int64

# Location computations tests

In [ ]:
import numpy as np

def calculate_haversine_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculates the great-circle distance between two points 
    on the Earth surface in kilometers using NumPy arrays.
    """
    # Earth radius in kilometers
    R = 6371.0 
    
    # Convert degrees to radians (NumPy trig functions require radians)
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    # Calculate differences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Apply the Haversine formula
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    
    # arcsin is mathematically equivalent to arctan2 for this and often slightly faster in numpy
    c = 2 * np.arcsin(np.sqrt(a)) 
    
    # Return distance in kilometers
    return R * c

1. Create spatial grid: distance between unique company locations and creator locations
2. Filter the creators that are within an eligible range (e.g., 25km)
    - Maybe explore whether I can come up with a metric that is a function of distance, e.g. weighted by distance from deal location
3. Per deal:
    - Total active (eligible) creators in last 7 days/30 days
        - Consider Pro creators? (creators with high follower count/ good reviews)


1. The Pre-Computed Spatial Grid (Calculate Once)

Do not calculate distances between deals and creators. Calculate distances between Unique Deal Locations and Unique Creator Locations.

Locations don't move. A coordinate in Amsterdam is always the same distance from a coordinate in Utrecht.

In [ ]:
# 1. Create a cross-join of ONLY the unique locations (1.3 million rows)
# df_unique_deals: 1710 rows (deal_loc_id, deal_lat, deal_lon)
# df_unique_creators: 773 rows (creator_loc_id, creator_lat, creator_lon)

df_spatial_grid = df_unique_deals.merge(df_unique_creators, how='cross')

# 2. Vectorize the Haversine distance formula here
# This gives you a permanent lookup table of distances between location IDs
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['deal_lat'], df_spatial_grid['deal_lon'],
    df_spatial_grid['creator_lat'], df_spatial_grid['creator_lon']
)

In [ ]:
# Calculate the distance for all 1.3 million combinations instantly
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['deal_lat'], 
    df_spatial_grid['deal_lon'],
    df_spatial_grid['creator_lat'], 
    df_spatial_grid['creator_lon']
)

# Optional: If you only care about creators within a specific radius (e.g., 50km), 
# drop the rest right now to save RAM before you join your time-series data!
df_spatial_grid = df_spatial_grid[df_spatial_grid['distance_km'] <= 50.0]

2. Dynamic Temporal Filtering (Calculate Often)

Now deal with the time aspect. You want to know who was active last week.§§

In [ ]:
# Filter your main creators dataframe (the one with 180k rows)
active_last_week = df_creators[df_creators['last_active'] >= '2026-02-12']

# Count how many ACTIVE creators are sitting at each unique location ID
# Result: A tiny dataframe of 773 rows showing available supply right now
supply_by_location = active_last_week.groupby('creator_loc_id').size().reset_index(name='active_creators')

3. The Final Join (Milliseconds)

Now, map that active supply onto your permanent spatial grid, and filter for deals that have creators within your desired radius (e.g., 25km).

In [ ]:
# Join the active counts to the spatial grid
eligible_supply = df_spatial_grid.merge(supply_by_location, on='creator_loc_id', how='left')

# Drop locations where no one is active
eligible_supply = eligible_supply.dropna(subset=['active_creators'])

# Filter for the radius you care about (e.g., within 25km)
deals_with_supply = eligible_supply[eligible_supply['distance_km'] <= 25]

# Group by deal to see total eligible creators nearby!
final_deal_supply = deals_with_supply.groupby('deal_loc_id')['active_creators'].sum()

In [6]:
df_locs = df_apps[~df_apps['company_location_id'].isna()]

In [11]:
creator_locs = df_locs[['longitude_creator', 'latitude_creator']]
partner_locs = df_locs[['longitude_partner', 'latitude_partner']]

In [13]:
partner_locs

,longitude_partner,latitude_partner
0,4.881908,52.358698
1,4.881908,52.358698
2,4.881908,52.358698
3,4.881908,52.358698
4,4.881908,52.358698
...,...,...
269727,4.887471,52.390218
269728,4.887471,52.390218
269729,4.887471,52.390218
269730,4.887471,52.390218


In [12]:
creator_locs

,longitude_creator,latitude_creator
0,5.055686,50.979461
1,4.904139,52.367573
2,4.414990,51.219930
3,4.904139,52.367573
4,NaN,NaN
...,...,...
269727,6.148591,52.291527
269728,6.148591,52.291527
269729,6.148591,52.291527
269730,5.122957,52.091925
